## 0 · Setup — clone repo, install deps

**Environment:** you're driving a **Colab GPU runtime from VS Code**. Code runs on the Colab VM; files land under `/content/HE-IFD` — view/download results from the **VS Code remote Explorer** (right-click ▸ Download). Make sure the runtime is **GPU** (T4 is fine). The repo is public (no token).

In [ ]:
import os
if not os.path.isdir("/content/HE-IFD"):
    !git clone -q https://github.com/hkanpak21/HE-IFD.git /content/HE-IFD
%cd /content/HE-IFD
!git pull -q origin master
# torch/torchvision are preinstalled on Colab; add the rest:
!pip -q install transformers datasets timm
!git log --oneline -1

In [ ]:
import torch
ok = torch.cuda.is_available()
print("CUDA:", ok, "|", torch.cuda.get_device_name(0) if ok else "NO GPU — Runtime ▸ Change runtime type ▸ T4 GPU")

# 027 · DP-MERF generator — soundness test + 2-cell re-verify

The old generator released **raw records**; the fix samples fresh points from a generator fit to the **DP** mean embedding. Mirrors `jobs/heifd_027_merf_verify.sh`.

**Gate / expected tell:** after the verify, Mode A (`dp_synth_all`) accuracy must **DROP at ε=2** (the old ~0.97 MNIST artifact gone), Mode B basin ≈ raw_union. Do **not** re-run the full 022 grid from here.

In [ ]:
# Output roots. If you mounted Drive above it already set CACHE_ROOT/RESULTS_ROOT;
# otherwise these local (Colab VM) defaults apply.
import os
DATA_ROOT = "data"
CACHE_ROOT = globals().get("CACHE_ROOT", "cache")
RESULTS_ROOT = globals().get("RESULTS_ROOT", "results")
print("DATA_ROOT=%s  CACHE_ROOT=%s  RESULTS_ROOT=%s" % (DATA_ROOT, CACHE_ROOT, RESULTS_ROOT))

### Step 1 — soundness pytest (gate). The verify is meaningful only if this passes.

In [ ]:
!python -m pytest tests/test_merf_dpsound.py -q

In [ ]:
import torchvision as tv
print("downloading datasets into", DATA_ROOT, "...")
tv.datasets.MNIST(DATA_ROOT, train=True, download=True)
tv.datasets.MNIST(DATA_ROOT, train=False, download=True)
tv.datasets.CIFAR100(DATA_ROOT, train=True, download=True)
tv.datasets.CIFAR100(DATA_ROOT, train=False, download=True)
print("vision datasets ready")

### Step 2 — 2-cell re-verify: {mlp_mnist, vit_b32_cifar100} × 5 methods, α=0.05
Mode A `dp_synth_all_eps{2,8}` · Mode B `merf_basin_eps{2,8}_K20` · ref `raw_union_K20`.

In [ ]:
!python -m src.sweep \
    --backbones mlp_mnist,vit_b32_cifar100 \
    --Ns 10 --alphas 0.05 \
    --methods dp_synth_all_eps2,dp_synth_all_eps8,merf_basin_eps2_K20,merf_basin_eps8_K20,raw_union_K20 \
    --seeds 42 --K 300 \
    --case heifd_027_merf_verify \
    --data-root $DATA_ROOT --cache-root $CACHE_ROOT --results-root $RESULTS_ROOT

### Results — accuracy by method (watch Mode A at ε=2)

In [ ]:
from pathlib import Path
rm = Path(RESULTS_ROOT) / "heifd_027_merf_verify" / "README.md"
print(rm.read_text() if rm.exists() else "(no README yet)")

In [ ]:
import json, glob
rows = []
for p in sorted(glob.glob(f"{RESULTS_ROOT}/heifd_027_merf_verify/cell_*.json")):
    d = json.load(open(p)); rows.append((d["backbone"], d["method"], d.get("acc"), d.get("status")))
for bb, m, acc, st in sorted(rows):
    print(f'{bb:18s} {m:22s} acc={acc if acc is None else round(acc,4)}  [{st}]')
print("\nEXPECTED: dp_synth_all_eps2 acc well BELOW the old ~0.97; merf_basin ≈ raw_union.")

In [ ]:
# Bundle results/heifd_027_merf_verify/ for retrieval. In VS Code you can instead just
# right-click results/heifd_027_merf_verify/ in the remote Explorer and Download.
import shutil
out = shutil.make_archive("/content/heifd_027_merf_verify_results", "zip", f"{RESULTS_ROOT}/heifd_027_merf_verify")
print("zipped ->", out, "\nDownload via the VS Code Explorer (right-click ▸ Download).")